# 🪞 Reflection Pattern

## What is Reflection?
An agent **reviews its own output** and improves it iteratively.

```
Generate → Critique → Revise → Critique → Revise → Done
```

## Use Cases
- Code review and improvement
- Essay writing and editing
- Mathematical proof checking
- Translation quality improvement

## Why Reflection Works
LLMs are better at CRITIQUE than GENERATION.
"Is this code correct?" is easier than "Write correct code."
Reflection exploits this asymmetry.


In [ ]:
# ── Reflection Agent ────────────────────────────────────────────────────
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

class CritiqueResult(BaseModel):
    is_satisfactory: bool = Field(description='Is the output good enough?')
    critique: str = Field(description='Specific feedback on what to improve')
    score: int = Field(description='Quality score 1-10', ge=1, le=10)

class ReflectionState(TypedDict):
    task: str
    current_draft: str
    iterations: int
    max_iterations: int

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)
critic_llm = llm.with_structured_output(CritiqueResult)

def generate(state: ReflectionState) -> dict:
    prompt = state['task']
    if state['current_draft']:
        prompt += f'\n\nPrevious draft (improve it):\n{state["current_draft"]}'
    response = llm.invoke([
        SystemMessage(content='You are an expert writer. Write clearly and concisely.'),
        HumanMessage(content=prompt)
    ])
    return {
        'current_draft': response.content,
        'iterations': state['iterations'] + 1
    }

def critique(state: ReflectionState) -> dict:
    result = critic_llm.invoke([
        SystemMessage(content='You are a strict editor. Be critical and specific.'),
        HumanMessage(content=f'Task: {state["task"]}\nDraft:\n{state["current_draft"]}\nCritique this.')
    ])
    print(f'Iteration {state["iterations"]} | Score: {result.score}/10 | {result.critique[:60]}...')
    return {'__critique__': result}  # temp storage — not in state schema

# Routing function needs access to critique result
_last_critique = None

def critique_and_route(state: ReflectionState) -> dict:
    global _last_critique
    result = critic_llm.invoke([
        SystemMessage(content='You are a strict editor.'),
        HumanMessage(content=f'Task: {state["task"]}\nDraft:\n{state["current_draft"]}\nCritique.')
    ])
    _last_critique = result
    print(f'Iter {state["iterations"]} | Score: {result.score}/10')
    return {}

def should_continue(state: ReflectionState) -> Literal['generate', '__end__']:
    if state['iterations'] >= state['max_iterations']:
        return '__end__'
    if _last_critique and _last_critique.is_satisfactory:
        return '__end__'
    return 'generate'

graph = StateGraph(ReflectionState)
graph.add_node('generate', generate)
graph.add_node('critique', critique_and_route)
graph.set_entry_point('generate')
graph.add_edge('generate', 'critique')
graph.add_conditional_edges('critique', should_continue)
app = graph.compile()

result = app.invoke({
    'task': 'Explain recursion to a 10-year-old in 3 sentences',
    'current_draft': '',
    'iterations': 0,
    'max_iterations': 3
})
print('\n=== FINAL DRAFT ===')
print(result['current_draft'])

## 🎯 Interview Questions

1. **What is the reflection pattern?**
2. **Why are LLMs better at critique than generation?**
3. **How do you prevent infinite reflection loops?**
4. **What metrics would you use to decide when to stop?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Reflection Pattern — Self-Improving Agents**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
